# Data 4 — Bovine Collagen Gelation Analysis
## Full Pipeline: Phases A, B, C, K, L, O, R, S, V

**Dataset:** Bovine collagen gelation at 3 concentrations (0.8%, 1.5%, 2.3% w/v) at 37°C  
**Files:** 20 Excel files — tiempo (time sweep) and frecuencia (frequency sweep) per replicate  
**Project:** Wavelet-Based Time-Frequency Analysis — Haemograph Pty Ltd  

### Important Note on Scope
This dataset contains **processed G'(t) and G''(t)** at ~1 Hz — NOT raw stress-strain waveforms.  
Phases requiring raw waveforms (FFT, CWT, harmonic detection, chirp analysis) belong on Data 1/2/3/6.  
This notebook focuses on what this dataset CAN contribute: gelation physics extraction, gel point detection,  
mutation number estimation, and concentration-dependent kinetics — the ground truth benchmarks for the pipeline.

## PHASE A — Data Integrity & Experimental Validation

In [ ]:
# ==========================================
# CELL A1: Imports
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
import warnings
from scipy import stats
from scipy.signal import butter, filtfilt
from scipy.stats import pearsonr, spearmanr
from statsmodels.tsa.stattools import adfuller, kpss
warnings.filterwarnings('ignore')

# Install if needed:
# !pip install statsmodels scipy pandas openpyxl matplotlib

print('All imports successful')

In [ ]:
# ==========================================
# CELL A2: Load all 20 Excel files
# ==========================================

# SET YOUR DATA FOLDER PATH HERE
data_folder_path = "/Users/riamishra/Desktop/MAST90106/code/Data4"

all_file_names = [
    f for f in os.listdir(data_folder_path)
    if f.endswith('.xlsx') and not f.startswith('~$')
]

time_sweep_columns = ['meas_pts','shear_stress_Pa','angular_freq_rad_s',
                      'storage_modulus_Pa','strain_pct','complex_viscosity_Pa_s',
                      'loss_modulus_Pa','temperature_C']
frequency_sweep_columns = ['meas_pts','angular_freq_rad_s','storage_modulus_Pa',
                            'loss_modulus_Pa','temperature_C']
col_counts = {'time_sweep': 8, 'frequency_sweep': 5}

data_dict = {}

for file_name in sorted(all_file_names):
    split_name = file_name.replace('.xlsx','').split('_')
    if len(split_name) < 5:
        print(f'Skipping: {file_name}')
        continue
    conc = split_name[2]
    rep  = split_name[3]
    dtype = split_name[4].replace('tiempo','time_sweep').replace('frecuencia','frequency_sweep')
    if conc not in data_dict: data_dict[conc] = {}
    if rep not in data_dict[conc]: data_dict[conc][rep] = {}
    fp = os.path.join(data_folder_path, file_name)
    try:
        df = pd.read_excel(fp, header=3, skiprows=[4], engine='openpyxl')
        df = df.iloc[:, :col_counts[dtype]]
        df.columns = time_sweep_columns if dtype=='time_sweep' else frequency_sweep_columns
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df = df.dropna(subset=['meas_pts']).reset_index(drop=True)
        data_dict[conc][rep][dtype] = df
    except Exception as e:
        print(f'Error loading {file_name}: {e}')

print('Loaded concentrations:', sorted(data_dict.keys()))
for conc in sorted(data_dict):
    for rep in sorted(data_dict[conc]):
        types = list(data_dict[conc][rep].keys())
        print(f'  Conc {conc}% | Rep {rep}: {types}')

In [ ]:
# ==========================================
# CELL A3: File Completeness Check
# ==========================================

print('=== FILE COMPLETENESS CHECK ===')
expected_types = {'time_sweep', 'frequency_sweep'}
all_ok = True
for conc in sorted(data_dict):
    for rep in sorted(data_dict[conc]):
        available = set(data_dict[conc][rep].keys())
        missing = expected_types - available
        if missing:
            print(f'  MISSING: Conc {conc}% Rep {rep} — missing {missing}')
            all_ok = False
        else:
            print(f'  OK: Conc {conc}% Rep {rep}')
if all_ok:
    print('\nAll files present and loaded correctly.')

In [ ]:
# ==========================================
# CELL A4: Shape and Size Validation
# ==========================================

print('=== SHAPE VALIDATION ===')
print(f'{"Conc":>6} {"Rep":>4} {"Type":>16} {"Rows":>6} {"Cols":>6}')
print('-' * 46)
for conc in sorted(data_dict):
    for rep in sorted(data_dict[conc]):
        for dtype in sorted(data_dict[conc][rep]):
            df = data_dict[conc][rep][dtype]
            flag = '' if df.shape[0] > 0 else ' ⚠️ EMPTY'
            print(f'{conc:>6} {rep:>4} {dtype:>16} {df.shape[0]:>6} {df.shape[1]:>6}{flag}')

In [ ]:
# ==========================================
# CELL A5: Unit and Range Consistency Check
# ==========================================

print('=== UNIT & RANGE CONSISTENCY ===')
for conc in sorted(data_dict):
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' in data_dict[conc][rep]:
            df = data_dict[conc][rep]['time_sweep']
            freq_vals = df['angular_freq_rad_s'].unique()
            temp_range = (df['temperature_C'].min(), df['temperature_C'].max())
            freq_ok = all(abs(f - 6.28) < 0.1 for f in freq_vals)
            temp_ok = 36 < temp_range[0] and temp_range[1] < 39
            print(f'Conc {conc}% Rep {rep}: freq={freq_vals[0]:.2f} rad/s {"✅" if freq_ok else "⚠️"} | temp={temp_range[0]:.1f}-{temp_range[1]:.1f}°C {"✅" if temp_ok else "⚠️"}')

In [ ]:
# ==========================================
# CELL A6: NaN and Missing Value Audit
# ==========================================

print('=== NaN AUDIT ===')
any_nan = False
for conc in sorted(data_dict):
    for rep in sorted(data_dict[conc]):
        for dtype in data_dict[conc][rep]:
            df = data_dict[conc][rep][dtype]
            nan_count = df.isnull().sum().sum()
            if nan_count > 0:
                print(f'  ⚠️ Conc {conc}% Rep {rep} {dtype}: {nan_count} NaN values')
                any_nan = True
if not any_nan:
    print('  ✅ No NaN values found in any dataset')

## PHASE B — Signal Diagnostics

In [ ]:
# ==========================================
# CELL B1: Stationarity Testing (ADF + KPSS) on G'(t)
# ==========================================
# ADF: H0 = non-stationary. p < 0.05 means stationary
# KPSS: H0 = stationary. p < 0.05 means non-stationary
# For gelation: G'(t) should be NON-stationary (rising over time)

print('=== STATIONARITY TEST ON G(t) ===')
print(f'{"Conc":>6} {"Rep":>4} {"ADF p":>8} {"KPSS p":>8} {"Verdict":>20}')
print('-' * 55)

for conc in sorted(data_dict):
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        G_prime = data_dict[conc][rep]['time_sweep']['storage_modulus_Pa'].values
        
        # ADF test
        adf_result = adfuller(G_prime, autolag='AIC')
        adf_p = adf_result[1]
        
        # KPSS test
        try:
            kpss_result = kpss(G_prime, regression='c', nlags='auto')
            kpss_p = kpss_result[1]
        except:
            kpss_p = None
        
        # Verdict: gelation = non-stationary = ADF p > 0.05, KPSS p < 0.05
        if adf_p > 0.05:
            verdict = 'Non-stationary ✅'
        else:
            verdict = 'Stationary ⚠️'
        
        kpss_str = f'{kpss_p:.3f}' if kpss_p else 'N/A'
        print(f'{conc:>6} {rep:>4} {adf_p:>8.3f} {kpss_str:>8} {verdict:>20}')

print('\nNote: Gelating G(t) should be non-stationary — confirms rising modulus over time.')
print('Non-stationary signal → Fourier assumes stationarity → Fourier is inappropriate here.')

In [ ]:
# ==========================================
# CELL B2: Signal Drift Detection
# ==========================================

print('=== DRIFT DETECTION (Linear trend in G(t)) ===')
print(f'{"Conc":>6} {"Rep":>4} {"Slope (Pa/step)":>16} {"R²":>6} {"Drift?":>10}')
print('-' * 50)

for conc in sorted(data_dict):
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['time_sweep']
        x = np.arange(len(df))
        y = df['storage_modulus_Pa'].values
        slope, intercept, r, p, se = stats.linregress(x, y)
        r2 = r**2
        drift = 'Yes (gelation)' if slope > 0 and p < 0.05 else 'No'
        print(f'{conc:>6} {rep:>4} {slope:>16.4f} {r2:>6.3f} {drift:>10}')

In [ ]:
# ==========================================
# CELL B3: Mutation Onset Detection (Early Stage)
# ==========================================
# Find the point where G'(t) starts rising above baseline
# Use 3-sigma rule: onset = first point > mean + 3*std of first 5 points

print('=== MUTATION ONSET DETECTION ===')
onset_results = {}

for conc in sorted(data_dict):
    onset_results[conc] = []
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        G = data_dict[conc][rep]['time_sweep']['storage_modulus_Pa'].values
        baseline_mean = np.mean(G[:5])
        baseline_std  = np.std(G[:5])
        threshold = baseline_mean + 3 * baseline_std
        onset_idx = next((i for i, v in enumerate(G) if v > threshold), None)
        onset_results[conc].append(onset_idx)
        print(f'Conc {conc}% Rep {rep}: baseline G={baseline_mean:.3f} Pa | onset at index {onset_idx}')

print('\nSummary — Mean onset index per concentration:')
for conc in sorted(onset_results):
    valid = [v for v in onset_results[conc] if v is not None]
    if valid:
        print(f'  Conc {conc}%: mean onset = {np.mean(valid):.1f} ± {np.std(valid):.1f} steps')

## PHASE C — Signal Energy & Information Content

In [ ]:
# ==========================================
# CELL C1: Shannon Entropy of G'(t) Evolution
# ==========================================
# Higher entropy = more disordered / less predictable = early gelation
# Lower entropy = more ordered / plateau = late gelation

def shannon_entropy(signal):
    signal = np.array(signal)
    signal = signal[~np.isnan(signal)]
    # normalise to probability distribution
    hist, _ = np.histogram(signal, bins=10, density=True)
    hist = hist[hist > 0]
    return -np.sum(hist * np.log2(hist + 1e-10))

print('=== SHANNON ENTROPY OF G(t) ===')
print(f'{"Conc":>6} {"Rep":>4} {"Entropy (G\')": >14} {"Entropy (G\'\')": >15}')
print('-' * 45)

entropy_results = {}
for conc in sorted(data_dict):
    entropy_results[conc] = []
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['time_sweep']
        H_Gp  = shannon_entropy(df['storage_modulus_Pa'].values)
        H_Gpp = shannon_entropy(df['loss_modulus_Pa'].values)
        entropy_results[conc].append(H_Gp)
        print(f'{conc:>6} {rep:>4} {H_Gp:>14.4f} {H_Gpp:>15.4f}')

In [ ]:
# ==========================================
# CELL C2: Energy Evolution — G'(t) cumulative energy
# ==========================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
concentrations = sorted(data_dict.keys())

for idx, conc in enumerate(concentrations):
    ax = axes[idx]
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        G = data_dict[conc][rep]['time_sweep']['storage_modulus_Pa'].values
        cumulative_energy = np.cumsum(G**2)
        ax.plot(cumulative_energy, label=f'Rep {rep}')
    ax.set_title(f'Cumulative Energy G(t)\nConc {conc}%')
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Cumulative Energy (Pa²)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Phase C: Cumulative G\' Energy Across Concentrations', fontweight='bold')
plt.tight_layout()
plt.show()

## PHASE K — Gelation Physics Extraction

In [ ]:
# ==========================================
# CELL K1: G'(t) and G''(t) Evolution — All Concentrations
# ==========================================

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
concentrations = sorted(data_dict.keys())

for col, conc in enumerate(concentrations):
    ax_top = axes[0, col]
    ax_bot = axes[1, col]
    
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['time_sweep']
        t = df['meas_pts'].values
        Gp  = df['storage_modulus_Pa'].values
        Gpp = df['loss_modulus_Pa'].values
        ax_top.plot(t, Gp,  label=f'Rep {rep}')
        ax_bot.plot(t, Gpp, label=f'Rep {rep}', linestyle='--')
    
    ax_top.set_title(f"G'(t) — Conc {conc}%")
    ax_top.set_xlabel('Measurement Point')
    ax_top.set_ylabel("G' (Pa)")
    ax_top.legend(fontsize=8)
    ax_top.grid(True, alpha=0.3)
    
    ax_bot.set_title(f"G''(t) — Conc {conc}%")
    ax_bot.set_xlabel('Measurement Point')
    ax_bot.set_ylabel("G'' (Pa)")
    ax_bot.legend(fontsize=8)
    ax_bot.grid(True, alpha=0.3)

plt.suptitle("Phase K: Storage (G') and Loss (G'') Modulus Evolution", fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# CELL K2: Gel Point Detection — G'/G'' Crossover
# ==========================================

print('=== GEL POINT DETECTION (G\'/G\'\' Crossover) ===')
gel_point_results = {}

for conc in sorted(data_dict):
    gel_point_results[conc] = []
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['time_sweep']
        Gp  = df['storage_modulus_Pa'].values
        Gpp = df['loss_modulus_Pa'].values
        diff = Gp - Gpp  # positive when G' > G'' (gel state)
        
        # find first sign change from negative to positive
        gel_idx = None
        for i in range(1, len(diff)):
            if diff[i-1] < 0 and diff[i] >= 0:
                gel_idx = i
                break
        
        gel_point_results[conc].append(gel_idx)
        state = 'Gel' if Gp[-1] > Gpp[-1] else 'Sol'
        print(f'Conc {conc}% Rep {rep}: gel point at step {gel_idx} | final state = {state}')

print('\nSummary:')
for conc in sorted(gel_point_results):
    valid = [v for v in gel_point_results[conc] if v is not None]
    if valid:
        print(f'  Conc {conc}%: gel point at step {np.mean(valid):.1f} ± {np.std(valid):.1f}')
    else:
        print(f'  Conc {conc}%: no crossover detected in time window')

In [ ]:
# ==========================================
# CELL K3: Gel Point Visualisation with Crossover Markers
# ==========================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for col, conc in enumerate(sorted(data_dict.keys())):
    ax = axes[col]
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['time_sweep']
        t   = df['meas_pts'].values
        Gp  = df['storage_modulus_Pa'].values
        Gpp = df['loss_modulus_Pa'].values
        
        ax.plot(t, Gp,  label=f"G' Rep {rep}")
        ax.plot(t, Gpp, linestyle='--', label=f"G'' Rep {rep}")
        
        # mark crossover
        diff = Gp - Gpp
        for i in range(1, len(diff)):
            if diff[i-1] < 0 and diff[i] >= 0:
                ax.axvline(x=t[i], color='red', linestyle=':', alpha=0.6, linewidth=1.5)
                ax.annotate('Gel Point', xy=(t[i], Gp[i]),
                            xytext=(t[i]+1, Gp[i]*1.2),
                            fontsize=7, color='red',
                            arrowprops=dict(arrowstyle='->', color='red', lw=0.8))
                break
    
    ax.set_title(f'Gel Point — Conc {conc}%')
    ax.set_xlabel('Measurement Point')
    ax.set_ylabel('Modulus (Pa)')
    ax.legend(fontsize=6)
    ax.grid(True, alpha=0.3)

plt.suptitle('Phase K: Gel Point Detection via G\'/G\'\' Crossover', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# CELL K4: Phase Angle (tan delta) Evolution
# ==========================================
# At gel point: tan(delta) = G''/G' should converge toward 1
# Below 1 = gel state (elastic dominant)
# Above 1 = sol state (viscous dominant)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for col, conc in enumerate(sorted(data_dict.keys())):
    ax = axes[col]
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['time_sweep']
        Gp  = df['storage_modulus_Pa'].values
        Gpp = df['loss_modulus_Pa'].values
        tan_delta = Gpp / (Gp + 1e-10)
        ax.plot(df['meas_pts'].values, tan_delta, label=f'Rep {rep}')
    
    ax.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5, label='Gel point (tan δ = 1)')
    ax.set_title(f'tan(δ) Evolution — Conc {conc}%')
    ax.set_xlabel('Measurement Point')
    ax.set_ylabel('tan(δ) = G\'\'/G\'')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, max(3, ax.get_ylim()[1]))

plt.suptitle('Phase K: Phase Angle (tan δ) Evolution — Gel Point at tan δ = 1', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# CELL K5: Winter-Chambon Power Law Exponent alpha
# ==========================================
# From frequency sweep: at gel point G' ~ G'' ~ omega^alpha
# Fit log(G') vs log(omega) from frequency sweep
# alpha = slope of log-log plot

print('=== WINTER-CHAMBON EXPONENT alpha (from frequency sweep) ===')
alpha_results = {}

for conc in sorted(data_dict):
    alpha_results[conc] = []
    for rep in sorted(data_dict[conc]):
        if 'frequency_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['frequency_sweep']
        
        # only use rows where both G' and G'' are positive
        df_valid = df[(df['storage_modulus_Pa'] > 0) & (df['loss_modulus_Pa'] > 0)].copy()
        if len(df_valid) < 3: continue
        
        log_omega = np.log(df_valid['angular_freq_rad_s'].values)
        log_Gp    = np.log(df_valid['storage_modulus_Pa'].values)
        log_Gpp   = np.log(df_valid['loss_modulus_Pa'].values)
        
        # fit slope
        slope_Gp,  _, r_Gp,  _, _ = stats.linregress(log_omega, log_Gp)
        slope_Gpp, _, r_Gpp, _, _ = stats.linregress(log_omega, log_Gpp)
        alpha_avg = (slope_Gp + slope_Gpp) / 2
        alpha_results[conc].append(alpha_avg)
        
        print(f'Conc {conc}% Rep {rep}: alpha(G\')={slope_Gp:.3f} | alpha(G\'\')={slope_Gpp:.3f} | avg={alpha_avg:.3f} | R²(G\')={r_Gp**2:.3f}')

print('\nSummary — Mean alpha per concentration:')
for conc in sorted(alpha_results):
    vals = alpha_results[conc]
    if vals:
        print(f'  Conc {conc}%: alpha = {np.mean(vals):.3f} ± {np.std(vals):.3f}')
print('\nNote: alpha = 0.5 → balanced gel. alpha < 0.5 → strong gel. alpha > 0.5 → weak gel.')
print('Reference: Hawkins et al. 2008 gelatin: alpha = 0.705')

In [ ]:
# ==========================================
# CELL K6: Frequency Sweep Power Law Log-Log Plots
# ==========================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for col, conc in enumerate(sorted(data_dict.keys())):
    ax = axes[col]
    for rep in sorted(data_dict[conc]):
        if 'frequency_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['frequency_sweep']
        df_valid = df[df['storage_modulus_Pa'] > 0]
        if len(df_valid) < 2: continue
        ax.loglog(df_valid['angular_freq_rad_s'], df_valid['storage_modulus_Pa'],
                  'o-', label=f"G' Rep {rep}", markersize=4)
        ax.loglog(df_valid['angular_freq_rad_s'], df_valid['loss_modulus_Pa'],
                  's--', label=f"G'' Rep {rep}", markersize=4)
    
    ax.set_title(f'Log-Log Frequency Sweep — Conc {conc}%')
    ax.set_xlabel('Angular Frequency (rad/s)')
    ax.set_ylabel('Modulus (Pa)')
    ax.legend(fontsize=6)
    ax.grid(True, which='both', alpha=0.3)

plt.suptitle('Phase K: Frequency Sweep — Power Law Scaling (slope = alpha)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# CELL K7: Plateau Modulus Comparison Across Concentrations
# ==========================================

print('=== PLATEAU MODULUS (G_inf) COMPARISON ===')
plateau_data = {}

for conc in sorted(data_dict):
    plateau_data[conc] = []
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        Gp = data_dict[conc][rep]['time_sweep']['storage_modulus_Pa'].values
        plateau = np.mean(Gp[-5:])  # average of last 5 points
        plateau_data[conc].append(plateau)
        print(f'  Conc {conc}% Rep {rep}: plateau G\' = {plateau:.3f} Pa')

print('\nSummary:')
for conc in sorted(plateau_data):
    vals = plateau_data[conc]
    print(f'  Conc {conc}%: G_inf = {np.mean(vals):.3f} ± {np.std(vals):.3f} Pa')

# Plot plateau vs concentration
fig, ax = plt.subplots(figsize=(6, 4))
concs = [float(c) for c in sorted(plateau_data.keys())]
means = [np.mean(plateau_data[c]) for c in sorted(plateau_data.keys())]
stds  = [np.std(plateau_data[c])  for c in sorted(plateau_data.keys())]
ax.errorbar(concs, means, yerr=stds, fmt='o-', capsize=5, linewidth=2, markersize=8)
ax.set_xlabel('Concentration (%)')
ax.set_ylabel('Plateau G\' (Pa)')
ax.set_title('Plateau Modulus vs Concentration')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# CELL K8: Gelation Rate dG'/dt
# ==========================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

gelation_rate_results = {}

for col, conc in enumerate(sorted(data_dict.keys())):
    ax = axes[col]
    max_rates = []
    
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['time_sweep']
        Gp = df['storage_modulus_Pa'].values
        t  = df['meas_pts'].values
        dGdt = np.gradient(Gp, t)
        max_rates.append(np.max(dGdt))
        ax.plot(t, dGdt, label=f'Rep {rep}')
    
    gelation_rate_results[conc] = max_rates
    ax.set_title(f'dG\'/dt — Conc {conc}%')
    ax.set_xlabel('Measurement Point')
    ax.set_ylabel("dG'/dt (Pa/step)")
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Phase K: Gelation Rate dG\'/dt Across Concentrations', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nMax gelation rate per concentration:')
for conc in sorted(gelation_rate_results):
    vals = gelation_rate_results[conc]
    print(f'  Conc {conc}%: max dG\'dt = {np.mean(vals):.4f} ± {np.std(vals):.4f} Pa/step')

In [ ]:
# ==========================================
# CELL K9: Complex Modulus |G*| and Phase Angle delta
# ==========================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for col, conc in enumerate(sorted(data_dict.keys())):
    ax = axes[col]
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['time_sweep']
        Gp  = df['storage_modulus_Pa'].values
        Gpp = df['loss_modulus_Pa'].values
        G_star = np.sqrt(Gp**2 + Gpp**2)
        delta  = np.degrees(np.arctan2(Gpp, Gp))
        t = df['meas_pts'].values
        
        l1, = ax.plot(t, G_star, label=f'|G*| Rep {rep}')
    
    ax2 = ax.twinx()
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['time_sweep']
        Gp  = df['storage_modulus_Pa'].values
        Gpp = df['loss_modulus_Pa'].values
        delta = np.degrees(np.arctan2(Gpp, Gp))
        ax2.plot(df['meas_pts'].values, delta, linestyle=':', alpha=0.6, label=f'δ Rep {rep}')
    
    ax.set_title(f'|G*| and δ — Conc {conc}%')
    ax.set_xlabel('Measurement Point')
    ax.set_ylabel('|G*| (Pa)')
    ax2.set_ylabel('Phase angle δ (°)')
    ax2.axhline(y=45, color='red', linestyle='--', linewidth=1, alpha=0.5, label='δ=45° (gel point)')
    ax.grid(True, alpha=0.3)

plt.suptitle('Phase K: Complex Modulus |G*| and Phase Angle δ Evolution', fontweight='bold')
plt.tight_layout()
plt.show()

## PHASE L — Mutation Number Framework

In [ ]:
# ==========================================
# CELL L1: Mutation Number N_mu Calculation
# ==========================================
# N_mu = (T/G) * (dG/dt)
# T = measurement period = 1/frequency = 1/1Hz = 1s (at 6.28 rad/s)
# Threshold: N_mu < 0.15 for LTI assumption to hold
# Mours & Winter 1994

T = 1.0 / 1.0  # period = 1s at 1 Hz (6.28 rad/s)
N_mu_threshold = 0.15

print('=== MUTATION NUMBER N_mu ===')
print(f'Measurement period T = {T} s | Threshold = {N_mu_threshold}')
print()

N_mu_results = {}

for conc in sorted(data_dict):
    N_mu_results[conc] = []
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['time_sweep']
        G  = df['storage_modulus_Pa'].values
        t  = df['meas_pts'].values
        
        # avoid division by zero
        G_safe = np.where(G < 0.01, 0.01, G)
        dGdt = np.gradient(G, t)
        N_mu = (T / G_safe) * np.abs(dGdt)
        
        N_mu_results[conc].append(N_mu)
        pct_valid = np.mean(N_mu < N_mu_threshold) * 100
        max_N_mu  = np.max(N_mu)
        
        status = '✅ PASS' if pct_valid >= 80 else '❌ FAIL'
        print(f'Conc {conc}% Rep {rep}: max N_mu = {max_N_mu:.4f} | {pct_valid:.1f}% below 0.15 | {status}')

print('\nNote: ≥80% of windows below N_mu = 0.15 required for Phase 1 pass (Q3 criterion)')

In [ ]:
# ==========================================
# CELL L2: Mutation Number Time Evolution Plot
# ==========================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for col, conc in enumerate(sorted(N_mu_results.keys())):
    ax = axes[col]
    for i, (rep, N_mu) in enumerate(zip(sorted(data_dict[conc].keys()), N_mu_results[conc])):
        t = data_dict[conc][rep]['time_sweep']['meas_pts'].values
        ax.plot(t, N_mu, label=f'Rep {rep}')
    
    ax.axhline(y=N_mu_threshold, color='red', linestyle='--',
               linewidth=2, label=f'Threshold = {N_mu_threshold}')
    ax.set_title(f'Mutation Number N_μ — Conc {conc}%')
    ax.set_xlabel('Measurement Point')
    ax.set_ylabel('N_μ')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, max(0.3, ax.get_ylim()[1]))

plt.suptitle('Phase L: Mutation Number N_μ — Threshold = 0.15 (Mours & Winter 1994)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# CELL L3: Mutation Number Violation Windows
# ==========================================

print('=== MUTATION NUMBER VIOLATION SUMMARY ===')
print('(Phase 1 criterion: ≥80% of windows must have N_mu < 0.15)\n')

for conc in sorted(N_mu_results):
    all_N_mu = np.concatenate(N_mu_results[conc])
    pct_valid = np.mean(all_N_mu < N_mu_threshold) * 100
    n_violations = np.sum(all_N_mu >= N_mu_threshold)
    status = '✅ PASS (Phase 1)' if pct_valid >= 80 else '❌ FAIL (Phase 1)'
    print(f'Conc {conc}%: {pct_valid:.1f}% valid | {n_violations} violations | {status}')

## PHASE O — Feature Stability & Robustness

In [ ]:
# ==========================================
# CELL O1: Replicate Consistency — CV Analysis
# ==========================================
# CV = coefficient of variation = std/mean * 100%
# Low CV = good reproducibility

print('=== REPLICATE CONSISTENCY — COEFFICIENT OF VARIATION ===')
print(f'{"Conc":>6} {"Metric":>22} {"Mean":>10} {"Std":>10} {"CV%":>8}')
print('-' * 60)

for conc in sorted(data_dict):
    plateau_vals = []
    gelpoint_vals = []
    max_rate_vals = []
    
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['time_sweep']
        Gp  = df['storage_modulus_Pa'].values
        Gpp = df['loss_modulus_Pa'].values
        t   = df['meas_pts'].values
        
        # plateau
        plateau_vals.append(np.mean(Gp[-5:]))
        
        # gel point
        diff = Gp - Gpp
        for i in range(1, len(diff)):
            if diff[i-1] < 0 and diff[i] >= 0:
                gelpoint_vals.append(t[i])
                break
        
        # max gelation rate
        dGdt = np.gradient(Gp, t)
        max_rate_vals.append(np.max(dGdt))
    
    def cv(vals):
        vals = np.array(vals)
        if len(vals) == 0 or np.mean(vals) == 0: return None
        return np.std(vals) / np.mean(vals) * 100
    
    for label, vals in [('Plateau G\' (Pa)', plateau_vals),
                         ('Gel Point Step', gelpoint_vals),
                         ('Max dG\'/dt', max_rate_vals)]:
        if vals:
            m, s, c = np.mean(vals), np.std(vals), cv(vals)
            flag = '✅' if c and c < 10 else '⚠️'
            print(f'{conc:>6} {label:>22} {m:>10.3f} {s:>10.3f} {c:>7.1f}% {flag}')

In [ ]:
# ==========================================
# CELL O2: Concentration Sensitivity Analysis
# ==========================================
# Does higher concentration consistently produce faster gelation and higher plateau?

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
concs = [float(c) for c in sorted(data_dict.keys())]

plateau_means, plateau_stds = [], []
gelpoint_means, gelpoint_stds = [], []
rate_means, rate_stds = [], []

for conc_str in sorted(data_dict.keys()):
    plateau_v, gp_v, rate_v = [], [], []
    for rep in sorted(data_dict[conc_str]):
        if 'time_sweep' not in data_dict[conc_str][rep]: continue
        df = data_dict[conc_str][rep]['time_sweep']
        Gp  = df['storage_modulus_Pa'].values
        Gpp = df['loss_modulus_Pa'].values
        t   = df['meas_pts'].values
        plateau_v.append(np.mean(Gp[-5:]))
        diff = Gp - Gpp
        for i in range(1, len(diff)):
            if diff[i-1] < 0 and diff[i] >= 0:
                gp_v.append(t[i])
                break
        dGdt = np.gradient(Gp, t)
        rate_v.append(np.max(dGdt))
    plateau_means.append(np.mean(plateau_v))
    plateau_stds.append(np.std(plateau_v))
    gelpoint_means.append(np.mean(gp_v) if gp_v else np.nan)
    gelpoint_stds.append(np.std(gp_v)  if gp_v else np.nan)
    rate_means.append(np.mean(rate_v))
    rate_stds.append(np.std(rate_v))

axes[0].errorbar(concs, plateau_means, yerr=plateau_stds, fmt='o-', capsize=5)
axes[0].set_xlabel('Concentration (%)')
axes[0].set_ylabel('Plateau G\' (Pa)')
axes[0].set_title('Plateau Modulus vs Concentration')
axes[0].grid(True, alpha=0.3)

axes[1].errorbar(concs, gelpoint_means, yerr=gelpoint_stds, fmt='s-', capsize=5, color='orange')
axes[1].set_xlabel('Concentration (%)')
axes[1].set_ylabel('Gel Point (step)')
axes[1].set_title('Gel Point Timing vs Concentration')
axes[1].grid(True, alpha=0.3)

axes[2].errorbar(concs, rate_means, yerr=rate_stds, fmt='^-', capsize=5, color='green')
axes[2].set_xlabel('Concentration (%)')
axes[2].set_ylabel('Max dG\'/dt (Pa/step)')
axes[2].set_title('Max Gelation Rate vs Concentration')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Phase O: Concentration Sensitivity Analysis', fontweight='bold')
plt.tight_layout()
plt.show()

## PHASE R — Algorithm Scoring System (for this dataset)

In [ ]:
# ==========================================
# CELL R1: Scoring Matrix for Data 4 Limitations
# ==========================================
# This cell explicitly documents WHY FFT/STFT/CWT cannot be properly
# applied to Data 4, and what CAN be extracted from it.

scoring = {
    'Criterion': [
        'Raw stress-strain waveforms',
        'Sampling rate ≥ 100 Hz',
        'OWCh chirp format',
        'Gel point detectable',
        'Winter-Chambon alpha extractable',
        'Mutation number calculable',
        'Multiple replicates',
        'Multiple concentrations',
        'FFT applicable on raw signal',
        'STFT applicable',
        'CWT applicable on raw signal',
        'Harmonic content extractable',
        'Ground truth gel time',
        'Openly accessible',
    ],
    'Data 4 Status': [
        '❌ Processed G\'(t) only',
        '❌ ~1 Hz only',
        '❌ Fixed freq SAOS',
        '✅ G\'/G\'\' crossover visible',
        '⚠️ Limited (21 freq pts)',
        '⚠️ From G\'(t) only',
        '✅ 3-4 per concentration',
        '✅ 0.8%, 1.5%, 2.3%',
        '❌ No raw waveform',
        '❌ No raw waveform',
        '❌ No raw waveform',
        '❌ No raw waveform',
        '✅ Crossover timing known',
        '✅ Zenodo open access',
    ],
    'Role in Pipeline': [
        'Need Data 1/2/3/6',
        'Need Data 1/2/3/6',
        'Need Data 1/2',
        'Ground truth benchmark',
        'Partial validation',
        'Approximate validation',
        'Statistical reliability',
        'Kinetic benchmarking',
        'Need Data 1/2/3/6',
        'Need Data 1/2/3/6',
        'Need Data 1/2/3/6',
        'Need Data 3/6',
        'Ground truth for Q1',
        'No barrier to access',
    ]
}

df_score = pd.DataFrame(scoring)
print(df_score.to_string(index=False))

## PHASE S — Failure Mode Analysis: Why Fourier Fails Here

In [ ]:
# ==========================================
# CELL S1: Demonstrating Fourier Limitation on G'(t)
# ==========================================
# If we apply FFT to G'(t), we get frequency components of the modulus EVOLUTION
# This is NOT the viscoelastic spectrum — it's meaningless for rheology
# This cell demonstrates WHY Fourier fails on non-stationary G'(t)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for col, conc in enumerate(sorted(data_dict.keys())):
    ax_top = axes[0, col]
    ax_bot = axes[1, col]
    
    df = data_dict[conc][sorted(data_dict[conc].keys())[0]]['time_sweep']
    Gp = df['storage_modulus_Pa'].values
    t  = df['meas_pts'].values
    
    # Show G'(t) — clearly non-stationary
    ax_top.plot(t, Gp, 'b-', linewidth=2)
    ax_top.set_title(f"G'(t) — Conc {conc}% (NON-STATIONARY)")
    ax_top.set_xlabel('Measurement Point')
    ax_top.set_ylabel("G' (Pa)")
    ax_top.grid(True, alpha=0.3)
    ax_top.text(0.05, 0.95, '⚠️ Fourier assumes stationarity\nThis signal violates LTI',
                transform=ax_top.transAxes, fontsize=8, va='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Apply FFT — result is meaningless for rheology
    fft_vals = np.abs(np.fft.fft(Gp))
    freqs = np.fft.fftfreq(len(Gp))
    mask = freqs > 0
    ax_bot.plot(freqs[mask], fft_vals[mask], 'r-')
    ax_bot.set_title(f'FFT of G\'(t) — NOT viscoelastic spectrum')
    ax_bot.set_xlabel('Normalised Frequency')
    ax_bot.set_ylabel('Magnitude')
    ax_bot.grid(True, alpha=0.3)
    ax_bot.text(0.05, 0.95, '❌ These are frequencies of\nmodulus evolution — NOT G(ω)',
                transform=ax_bot.transAxes, fontsize=8, va='top',
                bbox=dict(boxstyle='round', facecolor='salmon', alpha=0.5))

plt.suptitle('Phase S: WHY Fourier Fails on Non-Stationary G\'(t) — Stationarity Violation',
             fontweight='bold', color='darkred')
plt.tight_layout()
plt.show()
print('Key insight: FFT of G\'(t) does NOT give G\'(omega). It gives frequency components')
print('of how the modulus evolved in time — a meaningless artifact for rheological analysis.')
print('This is the fundamental limitation of Fourier on evolving materials.')

In [ ]:
# ==========================================
# CELL S2: Mutation Number as Quantitative Proof of Fourier Failure
# ==========================================
# If N_mu > 0.15, Fourier will average over a structural change
# and produce a spectrum that doesn't represent any single material state

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['blue', 'orange', 'green']
for i, conc in enumerate(sorted(data_dict.keys())):
    all_N_mu = []
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['time_sweep']
        G  = df['storage_modulus_Pa'].values
        t  = df['meas_pts'].values
        G_safe = np.where(G < 0.01, 0.01, G)
        dGdt = np.gradient(G, t)
        N_mu = (T / G_safe) * np.abs(dGdt)
        all_N_mu.append(N_mu)
    
    N_mu_mean = np.mean(all_N_mu, axis=0)
    N_mu_std  = np.std(all_N_mu, axis=0)
    t_axis = np.arange(len(N_mu_mean))
    
    ax.plot(t_axis, N_mu_mean, color=colors[i], label=f'Conc {conc}%', linewidth=2)
    ax.fill_between(t_axis, N_mu_mean - N_mu_std, N_mu_mean + N_mu_std,
                    alpha=0.2, color=colors[i])

ax.axhline(y=0.15, color='red', linestyle='--', linewidth=2,
           label='Fourier validity threshold (N_μ = 0.15)')
ax.fill_between(range(45), 0.15, ax.get_ylim()[1] if ax.get_ylim()[1] > 0.15 else 0.5,
                alpha=0.1, color='red', label='Fourier FAILS in this region')

ax.set_xlabel('Measurement Point')
ax.set_ylabel('Mutation Number N_μ')
ax.set_title('Phase S: Mutation Number — Quantitative Proof of Fourier Limitation\n'
             'Above threshold: Fourier produces averaged, incorrect spectrum', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## PHASE V — Summary Visualisation

In [ ]:
# ==========================================
# CELL V1: Master Summary Dashboard
# ==========================================

fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(3, 3, figure=fig)

concentrations = sorted(data_dict.keys())
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# Row 1: G'(t) all concentrations overlaid
ax1 = fig.add_subplot(gs[0, :])
for i, conc in enumerate(concentrations):
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['time_sweep']
        label = f'{conc}% Rep {rep}' if rep == sorted(data_dict[conc].keys())[0] else None
        ax1.plot(df['meas_pts'].values, df['storage_modulus_Pa'].values,
                 color=colors[i % len(colors)], alpha=0.7, label=label)
ax1.set_title("G'(t) All Concentrations and Replicates", fontsize=12, fontweight='bold')
ax1.set_xlabel('Measurement Point')
ax1.set_ylabel("G' (Pa)")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Row 2 Col 1: Gel point timing
ax2 = fig.add_subplot(gs[1, 0])
for i, conc in enumerate(concentrations):
    gps = []
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['time_sweep']
        Gp, Gpp, t = df['storage_modulus_Pa'].values, df['loss_modulus_Pa'].values, df['meas_pts'].values
        diff = Gp - Gpp
        for j in range(1, len(diff)):
            if diff[j-1] < 0 and diff[j] >= 0:
                gps.append(t[j])
                break
    if gps:
        ax2.bar(float(conc), np.mean(gps), yerr=np.std(gps), capsize=5,
                color=colors[i], alpha=0.8, label=f'{conc}%')
ax2.set_title('Gel Point Timing', fontweight='bold')
ax2.set_xlabel('Concentration (%)')
ax2.set_ylabel('Gel Point (step)')
ax2.grid(True, alpha=0.3)

# Row 2 Col 2: Plateau modulus
ax3 = fig.add_subplot(gs[1, 1])
for i, conc in enumerate(concentrations):
    plateaus = [np.mean(data_dict[conc][rep]['time_sweep']['storage_modulus_Pa'].values[-5:])
                for rep in sorted(data_dict[conc]) if 'time_sweep' in data_dict[conc][rep]]
    ax3.bar(float(conc), np.mean(plateaus), yerr=np.std(plateaus), capsize=5,
            color=colors[i], alpha=0.8)
ax3.set_title('Plateau G\' (G_inf)', fontweight='bold')
ax3.set_xlabel('Concentration (%)')
ax3.set_ylabel('Plateau G\' (Pa)')
ax3.grid(True, alpha=0.3)

# Row 2 Col 3: Alpha values
ax4 = fig.add_subplot(gs[1, 2])
for i, conc in enumerate(concentrations):
    alphas = []
    for rep in sorted(data_dict[conc]):
        if 'frequency_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['frequency_sweep']
        df_v = df[df['storage_modulus_Pa'] > 0]
        if len(df_v) < 3: continue
        slope, *_ = stats.linregress(np.log(df_v['angular_freq_rad_s']),
                                     np.log(df_v['storage_modulus_Pa']))
        alphas.append(slope)
    if alphas:
        ax4.bar(float(conc), np.mean(alphas), yerr=np.std(alphas), capsize=5,
                color=colors[i], alpha=0.8)
ax4.axhline(y=0.5, color='red', linestyle='--', linewidth=1, label='α=0.5 (balanced)')
ax4.set_title('Winter-Chambon Exponent α', fontweight='bold')
ax4.set_xlabel('Concentration (%)')
ax4.set_ylabel('α (power law slope)')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)

# Row 3: Mutation Number summary
ax5 = fig.add_subplot(gs[2, :])
for i, conc in enumerate(concentrations):
    all_N = []
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['time_sweep']
        G = np.where(df['storage_modulus_Pa'].values < 0.01, 0.01,
                     df['storage_modulus_Pa'].values)
        dGdt = np.gradient(df['storage_modulus_Pa'].values, df['meas_pts'].values)
        all_N.append((T / G) * np.abs(dGdt))
    N_mean = np.mean(all_N, axis=0)
    ax5.plot(np.arange(len(N_mean)), N_mean, color=colors[i], label=f'Conc {conc}%', linewidth=2)
ax5.axhline(y=0.15, color='red', linestyle='--', linewidth=2, label='N_μ = 0.15 threshold')
ax5.set_title('Mutation Number N_μ — LTI Validity Check', fontweight='bold')
ax5.set_xlabel('Measurement Point')
ax5.set_ylabel('N_μ')
ax5.legend()
ax5.grid(True, alpha=0.3)

plt.suptitle('DATA 4 — Bovine Collagen Gelation: Complete Analysis Dashboard',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# CELL V2: Final Summary Table
# ==========================================

print('=' * 65)
print('DATA 4 — BOVINE COLLAGEN GELATION — ANALYSIS SUMMARY')
print('=' * 65)

summary = []
for conc in sorted(data_dict.keys()):
    plateaus, gps, rates, alphas, N_mu_pcts = [], [], [], [], []
    for rep in sorted(data_dict[conc]):
        if 'time_sweep' not in data_dict[conc][rep]: continue
        df = data_dict[conc][rep]['time_sweep']
        Gp, Gpp, t = (df['storage_modulus_Pa'].values,
                      df['loss_modulus_Pa'].values,
                      df['meas_pts'].values)
        plateaus.append(np.mean(Gp[-5:]))
        diff = Gp - Gpp
        for i in range(1, len(diff)):
            if diff[i-1] < 0 and diff[i] >= 0:
                gps.append(t[i])
                break
        rates.append(np.max(np.gradient(Gp, t)))
        G_safe = np.where(Gp < 0.01, 0.01, Gp)
        N_mu = (T / G_safe) * np.abs(np.gradient(Gp, t))
        N_mu_pcts.append(np.mean(N_mu < 0.15) * 100)
    
    if 'frequency_sweep' in data_dict[conc][sorted(data_dict[conc].keys())[0]]:
        df_f = data_dict[conc][sorted(data_dict[conc].keys())[0]]['frequency_sweep']
        df_v = df_f[df_f['storage_modulus_Pa'] > 0]
        if len(df_v) >= 3:
            slope, *_ = stats.linregress(np.log(df_v['angular_freq_rad_s']),
                                         np.log(df_v['storage_modulus_Pa']))
            alphas.append(slope)
    
    summary.append({
        'Conc (%)': conc,
        'n_reps': len(plateaus),
        'Plateau G\' (Pa)': f'{np.mean(plateaus):.2f}±{np.std(plateaus):.2f}',
        'Gel Point (step)': f'{np.mean(gps):.1f}±{np.std(gps):.1f}' if gps else 'N/D',
        'Max dG\'/dt': f'{np.mean(rates):.4f}',
        'alpha': f'{np.mean(alphas):.3f}' if alphas else 'N/D',
        'N_mu<0.15 (%)': f'{np.mean(N_mu_pcts):.1f}%',
        'Phase 1': '✅' if np.mean(N_mu_pcts) >= 80 else '❌'
    })

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))
print('\n✅ = N_mu criterion passes Phase 1 (≥80% windows below 0.15)')